In [1]:
import os
import time
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from requests.exceptions import RequestException, HTTPError
from bs4 import BeautifulSoup

In [ ]:

def housing_auto_save_scrape(start_page=1, total_pages=500, checkpoint_every_pages=20):
    csv_filename = 'housing.csv'
    current_batch = []
    total_saved_rows = 0

    if os.path.exists(csv_filename):
        try:
            existing_df = pd.read_csv(csv_filename)
            total_saved_rows = len(existing_df)
            print(f'Existing dataset detected.\nFound {total_saved_rows} rows already saved.')
        except:
            pass

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36",
        "Accept-Encoding": "gzip, deflate",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "DNT": "1",
        "Connection": "close",
        "Upgrade-Insecure-Requests": "1"
    }

    print(f"Initializing Private Property Native Engine (Target: Pages {start_page} to {start_page + total_pages - 1})...")
    print('=' * 70)

    for page_num in range(start_page, start_page + total_pages):
        page_url = f'https://privateproperty.ng/property-for-rent?page={page_num}'
        print(f'Cooking Master List | Page {page_num}...')

        try:
            # Native connection call
            response = requests.get(page_url, headers=headers, timeout=120)

            if response.status_code != 200:
                print(f"Server returned unexpected code {response.status_code} on page {page_num}")
                if current_batch:
                    total_saved_rows = save_checkpoint(current_batch, csv_filename)
                    current_batch = []
                break

            page_content = response.text
            soup = BeautifulSoup(page_content, 'html.parser')

            # MATCHED: Pulling down every identical layout element signature on the page
            listings = soup.find_all('div', class_='similar-listings-item')

            if not listings:
                print(f"Layout limit or firewall block encountered at page {page_num}.")
                break

            page_extracted_count = 0

            for item in listings:
                try:
                    # 1. Title Extraction
                    title_tag = item.find('div', class_='similar-listings-info').find('h2') if item.find('div', class_='similar-listings-info') else None
                    title = title_tag.get_text(strip=True) if title_tag else ""

                    # 2. Subtitle / Type Description
                    desc_tag = item.find('div', class_='similar-listings-info').find('h3') if item.find('div', class_='similar-listings-info') else None
                    description = desc_tag.get_text(strip=True) if desc_tag else ""

                    # 3. Address Extraction (Clean strip completely removes the SVG map marker)
                    loc_p = item.find('p', class_='listings-location')
                    address = loc_p.get_text(strip=True) if loc_p else "Unknown Location"

                    # 4. Price Extraction
                    price_div = item.find('div', class_='similar-listings-price')
                    price_tag = price_div.find('h4') if price_div else None
                    price = price_tag.get_text(strip=True) if price_tag else ""

                    # 5. Features Extraction (Directly pulling structural metrics safely)
                    benefit_ul = item.find('ul', class_='property-benefit')
                    if benefit_ul:
                        benefit_lis = benefit_ul.find_all('li')
                        beds = benefit_lis[0].get_text(strip=True) if len(benefit_lis) > 0 else "0"
                        baths = benefit_lis[1].get_text(strip=True) if len(benefit_lis) > 1 else "0"
                        toilets = benefit_lis[2].get_text(strip=True) if len(benefit_lis) > 2 else "0"
                    else:
                        beds, baths, toilets = "0", "0", "0"

                    # 6. Date Info Entry
                    date_tag = item.find('h5', class_='mt-0')
                    date_text = date_tag.get_text(strip=True) if date_tag else ""

                    current_batch.append({
                        'Title': title,
                        'Description': description,
                        'Address': address,
                        'Price': price,
                        'Beds': beds,
                        'Baths': baths,
                        'Toilets': toilets,
                        'Date_Info': date_text
                    })
                    page_extracted_count += 1

                except Exception as inner_e:
                    continue

            print(f"Page {page_num} compiled successfully. Parsed {page_extracted_count} listing items.")

            # Autosave Checkpoint Routine
            if page_num % checkpoint_every_pages == 0 and current_batch:
                print(f"\n[AUTOSAVE] Writing batch to disk at page {page_num}...")
                total_saved_rows = save_checkpoint(current_batch, csv_filename)
                current_batch = []
                print(f"Database sync complete. Total rows: {total_saved_rows}\n" + "="*50)

            # Crucial: Human scroll interval delay
            time.sleep(random.uniform(4.0, 7.5))

        except Exception as e:
            print(f"An exception stopped the processing queue on page {page_num}: {e}")
            break

    if current_batch:
        total_saved_rows = save_checkpoint(current_batch, csv_filename)
        print(f"Pipeline flush completed. Final dataset registry size: {total_saved_rows} rows.")

In [ ]:
def save_checkpoint(batch_list, filename):
  """
  Helper function to cleanly write or append data to the CSV file
  """

  if not batch_list:
    return len(pd.read_csv(filename)) if os.path.exists(filename) else 0

  new_data = pd.DataFrame(batch_list)

  # If file doesn't exist, create it with headers. If it does, append without headers.
  if not os.path.exists(filename):
      new_data.to_csv(filename, index=False)
  else:
      new_data.to_csv(filename, mode='a', header=False, index=False)

  # Return total current rows for logging
  return len(pd.read_csv(filename))

In [ ]:
housing_auto_save_scrape(start_page = 1,  total_pages = 500, checkpoint_every_pages = 20)

Existing dataset detected.
Found 22 rows already saved.
Initializing Private Property Native Engine (Target: Pages 1 to 500)...
Cooking Master List | Page 1...
Page 1 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 2...
Page 2 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 3...
Page 3 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 4...
Page 4 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 5...
Page 5 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 6...
Page 6 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 7...
Page 7 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 8...
Page 8 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 9...
Page 9 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 10...
Page 10 compiled successfully. Parsed 22 listing items.


In [ ]:
housing_auto_save_scrape(start_page = 305,  total_pages = 500, checkpoint_every_pages = 20)

Existing dataset detected.
Found 6710 rows already saved.
Initializing Private Property Native Engine (Target: Pages 305 to 804)...
Cooking Master List | Page 305...
Page 305 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 306...
Page 306 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 307...
Page 307 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 308...
Page 308 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 309...
Page 309 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 310...
Page 310 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 311...
Page 311 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 312...
Page 312 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 313...
Page 313 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 314...
Page 314 compil

In [ ]:
def save_checkpoint(batch_list, filename):
  """
  Helper function to cleanly write or append data to the CSV file
  """

  if not batch_list:
    return len(pd.read_csv(filename)) if os.path.exists(filename) else 0

  new_data = pd.DataFrame(batch_list)

  # If file doesn't exist, create it with headers. If it does, append without headers.
  if not os.path.exists(filename):
      new_data.to_csv(filename, index=False)
  else:
      new_data.to_csv(filename, mode='a', header=False, index=False)

  # Return total current rows for logging
  return len(pd.read_csv(filename))





def pp_housing_auto_save_scrape(start_page=1, total_pages=500, checkpoint_every_pages=20):
    csv_filename = 'propertypro.csv'
    current_batch = []
    total_saved_rows = 0

    if os.path.exists(csv_filename):
        try:
            existing_df = pd.read_csv(csv_filename)
            total_saved_rows = len(existing_df)
            print(f'Existing dataset detected.\nFound {total_saved_rows} rows already saved.')
        except:
            pass

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36",
        "Accept-Encoding": "gzip, deflate",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "DNT": "1",
        "Connection": "close",
        "Upgrade-Insecure-Requests": "1"
    }

    print(f"Initializing Private Property Native Engine (Target: Pages {start_page} to {start_page + total_pages - 1})...")
    print('=' * 70)

    for page_num in range(start_page, start_page + total_pages):
        page_url = f'https://propertypro.ng/property-for-rent/flat-apartment?page={page_num}'
        print(f'Cooking Master List | Page {page_num}...')

        try:
            # Native connection call
            response = requests.get(page_url, headers=headers, timeout=120)

            if response.status_code != 200:
                print(f"Server returned unexpected code {response.status_code} on page {page_num}")
                if current_batch:
                    total_saved_rows = save_checkpoint(current_batch, csv_filename)
                    current_batch = []
                break

            page_content = response.text
            soup = BeautifulSoup(page_content, 'html.parser')

            # MATCHED: Pulling down every identical layout element signature on the page
            listings = soup.find_all('div', class_='property-listing-content')

            if not listings:
                print(f"Layout limit or firewall block encountered at page {page_num}.")
                break

            page_extracted_count = 0

            for item in listings:
                try:
                    # 1. Title Extraction
                    title_tag = item.find('div', class_='pl-title').find('h3') if item.find('div', class_='pl-title') else None
                    title = title_tag.get_text(strip=True) if title_tag else ""

                    # 2. Subtitle / Type Description
                    desc_tag = item.find('div', class_='pl-title').find('h6') if item.find('div', class_='pl-title') else None
                    description = desc_tag.get_text(strip=True) if desc_tag else ""

                    # 3. Address Extraction (Clean strip completely removes the SVG map marker)
                    loc_p = item.find('div', class_='pl-title').find('p') if item.find('div', class_='pl-title') else None
                    address = loc_p.get_text(strip=True) if loc_p else "Unknown Location"

                    # 4. Price Extraction
                    price_div = item.find('div', class_='pl-price')
                    price_tag = price_div.find('h3') if price_div else None
                    price = price_tag.get_text(strip=True) if price_tag else ""

                    # 5. Feature Extraction (Clean strip completely removes the SVG map marker)
                    feature_div = item.find('div', class_='pl-price').find('h6') if item.find('div', class_='pl-price') else None
                    feature = feature_div.get_text(strip=True) if feature_div else "Unknown Location"

                    # 6. Date Info Entry
                    date_tag = item.find('p', class_='date-added')
                    date_text = date_tag.get_text(strip=True) if date_tag else ""

                    current_batch.append({
                        'Title': title,
                        'Description': description,
                        'Address': address,
                        'Price': price,
                        'Feature': feature,
                        'Date_Info': date_text
                    })
                    page_extracted_count += 1

                except Exception as inner_e:
                    continue

            print(f"Page {page_num} compiled successfully. Parsed {page_extracted_count} listing items.")

            # Autosave Checkpoint Routine
            if page_num % checkpoint_every_pages == 0 and current_batch:
                print(f"\n[AUTOSAVE] Writing batch to disk at page {page_num}...")
                total_saved_rows = save_checkpoint(current_batch, csv_filename)
                current_batch = []
                print(f"Database sync complete. Total rows: {total_saved_rows}\n" + "="*50)

            # Crucial: Human scroll interval delay
            time.sleep(random.uniform(4.0, 7.5))

        except Exception as e:
            print(f"An exception stopped the processing queue on page {page_num}: {e}")
            break

    if current_batch:
        total_saved_rows = save_checkpoint(current_batch, csv_filename)
        print(f"Pipeline flush completed. Final dataset registry size: {total_saved_rows} rows.")


pp_housing_auto_save_scrape(start_page = 1,  total_pages = 500, checkpoint_every_pages = 20)

Initializing Private Property Native Engine (Target: Pages 1 to 500)...
Cooking Master List | Page 1...
Page 1 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 2...
Page 2 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 3...
Page 3 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 4...
Page 4 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 5...
Page 5 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 6...
Page 6 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 7...
Page 7 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 8...
Page 8 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 9...
Page 9 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 10...
Page 10 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 11...
Page 11 compiled succes

In [ ]:
def save_checkpoint(batch_list, filename):
  """
  Helper function to cleanly write or append data to the CSV file
  """

  if not batch_list:
    return len(pd.read_csv(filename)) if os.path.exists(filename) else 0

  new_data = pd.DataFrame(batch_list)

  # If file doesn't exist, create it with headers. If it does, append without headers.
  if not os.path.exists(filename):
      new_data.to_csv(filename, index=False)
  else:
      new_data.to_csv(filename, mode='a', header=False, index=False)

  # Return total current rows for logging
  return len(pd.read_csv(filename))





def pp_housing_auto_save_scrape(start_page=1, total_pages = 800, checkpoint_every_pages=20):
    csv_filename = 'propertypro.csv'
    current_batch = []
    total_saved_rows = 0

    if os.path.exists(csv_filename):
        try:
            existing_df = pd.read_csv(csv_filename)
            total_saved_rows = len(existing_df)
            print(f'Existing dataset detected.\nFound {total_saved_rows} rows already saved.')
        except:
            pass

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36",
        "Accept-Encoding": "gzip, deflate",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "DNT": "1",
        "Connection": "close",
        "Upgrade-Insecure-Requests": "1"
    }

    print(f"Initializing Private Property Native Engine (Target: Pages {start_page} to {start_page + total_pages - 1})...")
    print('=' * 70)

    for page_num in range(start_page, start_page + total_pages):
        page_url = f'https://propertypro.ng/property-for-rent/flat-apartment?page={page_num}'
        print(f'Cooking Master List | Page {page_num}...')

        try:
            # Native connection call
            response = requests.get(page_url, headers=headers, timeout=120)

            if response.status_code != 200:
                print(f"Server returned unexpected code {response.status_code} on page {page_num}")
                if current_batch:
                    total_saved_rows = save_checkpoint(current_batch, csv_filename)
                    current_batch = []
                break

            page_content = response.text
            soup = BeautifulSoup(page_content, 'html.parser')

            # MATCHED: Pulling down every identical layout element signature on the page
            listings = soup.find_all('div', class_='property-listing-content')

            if not listings:
                print(f"Layout limit or firewall block encountered at page {page_num}.")
                break

            page_extracted_count = 0

            for item in listings:
                try:
                    # 1. Title Extraction
                    title_tag = item.find('div', class_='pl-title').find('h3') if item.find('div', class_='pl-title') else None
                    title = title_tag.get_text(strip=True) if title_tag else ""

                    # 2. Subtitle / Type Description
                    desc_tag = item.find('div', class_='pl-title').find('h6') if item.find('div', class_='pl-title') else None
                    description = desc_tag.get_text(strip=True) if desc_tag else ""

                    # 3. Address Extraction (Clean strip completely removes the SVG map marker)
                    loc_p = item.find('div', class_='pl-title').find('p') if item.find('div', class_='pl-title') else None
                    address = loc_p.get_text(strip=True) if loc_p else "Unknown Location"

                    # 4. Price Extraction
                    price_div = item.find('div', class_='pl-price')
                    price_tag = price_div.find('h3') if price_div else None
                    price = price_tag.get_text(strip=True) if price_tag else ""

                    # 5. Feature Extraction (Clean strip completely removes the SVG map marker)
                    feature_div = item.find('div', class_='pl-price').find('h6') if item.find('div', class_='pl-price') else None
                    feature = feature_div.get_text(strip=True) if feature_div else "Unknown Location"

                    # 6. Date Info Entry
                    date_tag = item.find('p', class_='date-added')
                    date_text = date_tag.get_text(strip=True) if date_tag else ""

                    current_batch.append({
                        'Title': title,
                        'Description': description,
                        'Address': address,
                        'Price': price,
                        'Feature': feature,
                        'Date_Info': date_text
                    })
                    page_extracted_count += 1

                except Exception as inner_e:
                    continue

            print(f"Page {page_num} compiled successfully. Parsed {page_extracted_count} listing items.")

            # Autosave Checkpoint Routine
            if page_num % checkpoint_every_pages == 0 and current_batch:
                print(f"\n[AUTOSAVE] Writing batch to disk at page {page_num}...")
                total_saved_rows = save_checkpoint(current_batch, csv_filename)
                current_batch = []
                print(f"Database sync complete. Total rows: {total_saved_rows}\n" + "="*50)

            # Crucial: Human scroll interval delay
            time.sleep(random.uniform(4.0, 7.5))

        except Exception as e:
            print(f"An exception stopped the processing queue on page {page_num}: {e}")
            break

    if current_batch:
        total_saved_rows = save_checkpoint(current_batch, csv_filename)
        print(f"Pipeline flush completed. Final dataset registry size: {total_saved_rows} rows.")


pp_housing_auto_save_scrape(start_page = 670,  total_pages = 800, checkpoint_every_pages = 20)

Existing dataset detected.
Found 17490 rows already saved.
Initializing Private Property Native Engine (Target: Pages 670 to 1469)...
Cooking Master List | Page 670...
Page 670 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 671...
Page 671 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 672...
Page 672 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 673...
Page 673 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 674...
Page 674 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 675...
Page 675 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 676...
Page 676 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 677...
Page 677 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 678...
Page 678 compiled successfully. Parsed 22 listing items.
Cooking Master List | Page 679...
Page 679 comp

In [2]:
try:
    data1 = pd.read_csv('nig_properties.csv')
    data2 = pd.read_csv('propertypro.csv')
except FileNotFoundError as f:
    print(f'Encountered a {f} trying to load from the local storage, meaning I\'m working from Colab..\nAttempting to load from the drive instead.')

    from google.colab import drive

    data1 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/lagos_properties/nig_properties.csv')
    data2 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/lagos_properties/propertypro.csv')
    print(f'Successfully loaded both datasets from Google cloud.\nLet\'s proceed.')

display(
    data1[data1.duplicated()].shape,
    data2[data2.duplicated()].shape
)

(553, 8)

(4722, 6)

In [3]:
display(
    data1[data1.duplicated()],
    data2[data2.duplicated()]
)

,Title,Description,Address,Price,Beds,Baths,Toilets,Date_Info
24,Unfurnished 3 Bedroom Apartment And A Bq,3 BEDROOM BLOCK OF FLATS FOR RENT,Fatai Arobieke Street Lekki Phase 1 Lekki Lagos,"₦23,000,000",3.0,3.0,4.0,Updated Today
25,2 Bedroom Flat,2 BEDROOM MINI FLAT FOR RENT,Mutual Estate Eputu Ibeju Lekki Lagos,"₦2,500,000",2.0,3.0,3.0,Updated Today
26,1234 Sqm Of Land With Building On It In Centra...,MIXED USE LAND FOR RENT,Off Admiralty Way Lekki Phase 1 Lekki Lagos,"₦100,000,000",NaN,NaN,NaN,Updated Today
27,Fully Furnished 4 Bedrooms Fully Detached Dupl...,4 BEDROOM DUPLEX FOR RENT,Orchid Lekki Lekki Phase 2 Lekki Lagos,"₦13,000,000",4.0,4.0,5.0,Updated Today
28,Self Compound 3 Bedrooms Semi Detached In Chev...,3 BEDROOM SEMI DETACHED DUPLEX FOR RENT,Chevron Lekki Chevron Drive Lekki Lagos,"₦9,000,000",3.0,3.0,4.0,Updated Today
...,...,...,...,...,...,...,...,...
7018,Luxury Ground Floor 3 Bedroom Apartment + Bq+p...,3 BEDROOM FLAT & APARTMENT For Rent,Old Ikoyi Lagos,"₦35,000,000",3.0,3.0,4.0,"Updated 21 Jun 2026, Added 06 May 2026"
7028,Luxury 4 Bedroom Terrace House On 3 Floors+ele...,4 BEDROOM TERRACED DUPLEX For Sale,Old Ikoyi Lagos,"₦1,100,000,000",4.0,4.0,5.0,"Updated 21 Jun 2026, Added 12 May 2026"
7040,Luxury 3 Bedroom Apartment +bq+pool In Banana ...,3 BEDROOM FLAT & APARTMENT For Rent,Banana Island Estate Banana Island Ikoyi Lagos,"₦35,000,000",3.0,3.0,4.0,"Updated 21 Jun 2026, Added 26 Apr 2026"
7062,Luxury 3 Bed Apartment On 2 Floors+bq+gym+pool...,3 BEDROOM MAISONETTE For Rent,Old Ikoyi Lagos,"₦35,000,000",3.0,3.0,4.0,"Updated 22 Jun 2026, Added 09 Jun 2026"


,Title,Description,Address,Price,Feature,Date_Info
32,2 Bedroom Waterfront Apartment,2 Bedroom Waterfront Apartment/Flat / Apartmen...,Osborne Foreshore Estate Ikoyi Lagos,"₦ 415,000,000",2 Beds 2 Baths,"Updated 21 Jun 2026, Added 02 Oct 2025"
42,Mini Flat,Mini Flat/Flat Apartment FOR Rent,Close To Alagbole Berger Ojodu Lagos,"₦ 1,500,000/year",NaN,"Updated 23 Jun 2026, Added 29 May 2026"
70,Mini Flat,Mini Flat/Flat Apartment FOR Rent,Close To Alagbole Berger Ojodu Lagos,"₦ 1,500,000/year",NaN,"Updated 23 Jun 2026, Added 29 May 2026"
75,Mini Flat,Mini Flat/Flat Apartment FOR Rent,Ibeju Lekki Ajah Lagos,"₦ 1,300,000/year",NaN,"Updated 23 Jun 2026, Added 29 May 2026"
88,3 Bedroom Apartment,3 Bedroom Apartment/Flat / Apartment FOR RENT,Old Ikoyi Ikoyi Lagos,"₦ 40,000,000/year",3 Beds 3 Baths,"Updated 21 Jun 2026, Added 27 Feb 2026"
...,...,...,...,...,...,...
17686,Furnished 1 Bedroom Flat In Orchid Lekki Pay &...,Furnished 1 Bedroom Flat In Orchid Lekki Pay &...,Orchid Lekki Lagos,"₦ 3,500,000/year",1 Beds 1 Baths,"Updated 18 May 2026, Added 11 May 2026"
17687,Luxury 2 Bedroom Townhome,Luxury 2 Bedroom Townhome/Flat Apartment FOR Rent,No 9 Samuel O. Oladele Crescent Katampe Main A...,"₦ 7,000,000/year",2 Beds 2 Baths,"Updated 02 May 2026, Added 19 Dec 2025"
17688,Newly Furnished/serviced 3 Bedroom Apartment F...,Newly Furnished/serviced 3 Bedroom Apartment F...,Jahi Abuja,"₦ 25,000,000/year",3 Beds 3 Baths,"Updated 17 Jun 2026, Added 20 Feb 2026"
17689,Furnished 1 Bedroom Luxury Apartment,Furnished 1 Bedroom Luxury Apartment/Flat Apar...,No 9 Samuel O. Oladele Crescent Katampe Main A...,"₦ 6,000,000/year",1 Beds 1 Baths,"Updated 16 Sep 2025, Added 30 Jul 2025"


In [4]:
data1 = data1.drop_duplicates()
data2 = data2.drop_duplicates()

In [5]:
display(
    data1.shape,
    data2.shape
)

(6529, 8)

(12969, 6)

In [6]:
display(
    data1.info(),
    data2.info()
)

<class 'pandas.core.frame.DataFrame'>
Index: 6529 entries, 0 to 7081
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        6529 non-null   object 
 1   Description  6529 non-null   object 
 2   Address      6529 non-null   object 
 3   Price        6529 non-null   object 
 4   Beds         5886 non-null   float64
 5   Baths        5530 non-null   float64
 6   Toilets      5530 non-null   float64
 7   Date_Info    6529 non-null   object 
dtypes: float64(3), object(5)
memory usage: 459.1+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 12969 entries, 0 to 17647
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Title        12969 non-null  object
 1   Description  12969 non-null  object
 2   Address      12969 non-null  object
 3   Price        12969 non-null  object
 4   Feature      11061 non-null  object
 5   Date_Info    12969 non-null  o

None

None

In [7]:
display(
    data1[data1.isnull().any(axis = 1)],
    data2[data2.isnull().any(axis = 1)]
)

,Title,Description,Address,Price,Beds,Baths,Toilets,Date_Info
3,1234 Sqm Of Land With Building On It In Centra...,MIXED USE LAND FOR RENT,Off Admiralty Way Lekki Phase 1 Lekki Lagos,"₦100,000,000",NaN,NaN,NaN,Updated Today
10,Fairmont Green & Smart Estate,RESIDENTIAL LAND For Sale,With Cofo Ibeju Lekki Lagos,"₦30,000,000",NaN,NaN,NaN,"Updated 22 Jun 2026, Added 09 Jun 2025"
73,100 Sqm Of Office Space On Ground Floor For Re...,OFFICE FOR RENT,Victoria Island Adeola Hopewell Victoria Islan...,"₦17,000,000",NaN,NaN,NaN,Updated Today
87,Spacious Room Self Contain With Modern Feature...,SELF CONTAIN FOR RENT,Off Costain Road Yaba Lagos,"₦1,500,000/year",NaN,1.0,1.0,"Updated 22 Jun 2026, Added 31 May 2026"
89,"Decent Room Self Contain With Modern Features,...",SELF CONTAIN FOR RENT,"New Garage, Close To Berger Oworonshoki Gbagad...","₦1,200,000/year",NaN,1.0,1.0,"Updated 22 Jun 2026, Added 31 May 2026"
...,...,...,...,...,...,...,...,...
7020,Massive Warehouse Space,10 BEDROOM WAREHOUSE FOR RENT,Wofun Olodo Rd Iwo Road Ibadan Oyo,"₦40,000,000/year",10.0,NaN,NaN,"Updated 22 Apr 2025, Added 29 Apr 2024"
7035,Massive Commercial Property On Large Grounds W...,OFFICE FOR RENT,Old Ikoyi Ikoyi Ikoyi Lagos,"₦30,000,000",NaN,NaN,NaN,"Updated 28 Oct 2023, Added 16 Oct 2021"
7038,Direct Clients Only Serviced Ultra Modern Offi...,OFFICE FOR RENT,Ikoyi Ikoyi Ikoyi Lagos,"₦235,000",NaN,NaN,NaN,"Updated 28 Oct 2023, Added 23 Jul 2021"
7051,Direct Cash Ready Clients 2 Units Of 18 Bed Ho...,HOTEL FOR RENT,Lekki 1 Lekki Phase 1 Lekki Lagos,"₦55,000,001",NaN,NaN,20.0,"Updated 28 Oct 2023, Added 08 Aug 2021"


,Title,Description,Address,Price,Feature,Date_Info
7,Mini Flat Upstairs,Mini Flat Upstairs/Flat Apartment FOR Rent,Lekki Lagos,"₦ 5,500,000/year",NaN,"Updated 23 Jun 2026, Added 12 Jun 2026"
13,Self Contain Upstairs,Self Contain Upstairs/Flat Apartment FOR Rent,Lekki Lagos,"₦ 2,500,000/year",NaN,"Updated 23 Jun 2026, Added 12 Jun 2026"
33,A Studio Apartment,A Studio Apartment/Flat Apartment FOR Rent,Lekki Phase 1 Lekki Lagos,"₦ 2,500,000/year",NaN,"Updated 23 Jun 2026, Added 05 Jun 2026"
40,2 Bedroom Flat,2 Bedroom Flat/Flat Apartment FOR Rent,Close To Alagbole Berger Ojodu Lagos,"₦ 1,800,000/year",NaN,"Updated 23 Jun 2026, Added 29 May 2026"
41,Mini Flat,Mini Flat/Flat Apartment FOR Rent,Close To Alagbole Berger Ojodu Lagos,"₦ 1,500,000/year",NaN,"Updated 23 Jun 2026, Added 29 May 2026"
...,...,...,...,...,...,...
17445,Room Self Contained,Room Self Contained/Flat Apartment FOR Rent,Ikotun Igando Lagos,"₦ 300,000/year",NaN,"Updated 14 Jun 2026, Added 10 Jan 2026"
17457,Room Self Contained,Room Self Contained/Flat Apartment FOR Rent,Fadeyi Shomolu Lagos,"₦ 800,000/year",NaN,"Updated 14 Jun 2026, Added 05 Jan 2026"
17463,2 Bedroom Flat,2 Bedroom Flat/Flat Apartment FOR Rent,Off Liasu Road Council Egbe Idimu Lagos,"₦ 2,000,000/year",NaN,"Updated 14 Jun 2026, Added 20 Dec 2025"
17482,Room Self Contained,Room Self Contained/Flat Apartment FOR Rent,Ocean Breeze Estate Ologolo Lekki Lagos,"₦ 1,500,000/year",NaN,"Updated 14 Jun 2026, Added 07 Apr 2026"


In [8]:
data1_features = ['Beds', 'Baths', 'Toilets']

for feature in data1[data1_features]:
    data1[feature] = data1[feature].fillna(0.0)
    data1[feature] = data1[feature].astype(int)

data1[['Bedrooms', 'Bathrooms']] = data1[['Beds', 'Baths']]
data1 = data1.drop(columns = ['Beds', 'Baths'])
data1.head(3)

,Title,Description,Address,Price,Toilets,Date_Info,Bedrooms,Bathrooms
0,Newly Renovated Room And Parlour Shortlet Apar...,1 BEDROOM MINI FLAT For Rent,11 Obamusa Agungi Lekki Lagos Agungi Lekki Lagos,"₦3,200,000/month",1,"Updated 21 Jun 2026, Added 10 Jun 2026",1,1
1,Unfurnished 3 Bedroom Apartment And A Bq,3 BEDROOM BLOCK OF FLATS FOR RENT,Fatai Arobieke Street Lekki Phase 1 Lekki Lagos,"₦23,000,000",4,Updated Today,3,3
2,2 Bedroom Flat,2 BEDROOM MINI FLAT FOR RENT,Mutual Estate Eputu Ibeju Lekki Lagos,"₦2,500,000",3,Updated Today,2,3


In [9]:
data2['Feature'] = data2['Feature'].fillna('0 0 0 0')

data2[['Bedrooms', 'None', 'Bathrooms', 'None']] = data2['Feature'].str.split(' ', n = 4, expand = True)
data2 = data2.drop(columns = 'None')

data2['Bathrooms'] = data2['Bathrooms'].fillna(0)
data2[['Bedrooms', 'Bathrooms']] = data2[['Bedrooms', 'Bathrooms']].astype(int)

data2[data2.isnull().any(axis = 1)]

,Title,Description,Address,Price,Feature,Date_Info,Bedrooms,Bathrooms


In [10]:
def clean_date(data_copy):
    data = data_copy.copy()
    data['Date_Info'] = data['Date_Info'].str.replace('Updated Today', 'Updated 23 Jun 2026, Added 23 Jun 2026')

    data[["Date Added", "Date Listed"]] = data["Date_Info"].str.split(",", n = 2, expand = True)

    data['Date Listed'] = data['Date Listed'].str.replace('Added ','').str.strip()
    data['Date Listed'] = pd.to_datetime(data['Date Listed'], format='%d %b %Y')
    data['Date Listed'] = data['Date Listed'].fillna(data['Date Listed'].mode()[0])

    data = data.drop(columns = ["Date_Info", "Date Added"])

    return data

data1 = clean_date(data1)
data2 = clean_date(data2)

display(
    data1.head(3),
    data2.head(3)
)

,Title,Description,Address,Price,Toilets,Bedrooms,Bathrooms,Date Listed
0,Newly Renovated Room And Parlour Shortlet Apar...,1 BEDROOM MINI FLAT For Rent,11 Obamusa Agungi Lekki Lagos Agungi Lekki Lagos,"₦3,200,000/month",1,1,1,2026-06-10
1,Unfurnished 3 Bedroom Apartment And A Bq,3 BEDROOM BLOCK OF FLATS FOR RENT,Fatai Arobieke Street Lekki Phase 1 Lekki Lagos,"₦23,000,000",4,3,3,2026-06-23
2,2 Bedroom Flat,2 BEDROOM MINI FLAT FOR RENT,Mutual Estate Eputu Ibeju Lekki Lagos,"₦2,500,000",3,2,3,2026-06-23


,Title,Description,Address,Price,Feature,Bedrooms,Bathrooms,Date Listed
0,Lovely Newly Renovated 2 Bedroom Flat,Lovely Newly Renovated 2 Bedroom Flat/Flat / A...,"Miracle Avenue, Opposite Wisebuyer Mall. Magbo...","₦ 1,500,000/year",2 Beds 1 Baths,2,1,2026-06-22
1,Ultra Luxury 3 Bedroom Apartment Bq Pool Gym,Ultra Luxury 3 Bedroom Apartment Bq Pool Gym/F...,Old Ikoyi Old Ikoyi Ikoyi Lagos,"₦ 54,000,000",3 Beds 3 Baths,3,3,2026-06-19
2,Mini Flat,Mini Flat/Flat Apartment FOR Rent,Lekki Phase 1 Lekki Lagos,"₦ 5,000,000/year",1 Beds 1 Baths,1,1,2026-03-07


In [11]:
# def clean_price(data_copy):
#     data = data_copy.copy()
#     data['Prices'] = data['Price'].str.split('/', n=1).str[0]

#     is_usd = data['Prices'].str.contains('$', regex = False, na = False)
#     data['Prices'] = (
#         data['Prices']
#             .str.replace('₦', '', regex = False)
#             .str.replace('$', '', regex = False)
#             .str.replace(',', '', regex = False)
#             .str.strip()
#     )

#     data['Prices'] = data['Prices'].astype(float)
#     is_usd_mask = is_usd & (data['Prices'] <= 10000)
#     data.loc[is_usd_mask, 'Prices'] *= 1366.92

#     data['Prices'] = data['Prices'].astype(int)
#     data = data[(data['Prices'] >= 100000)]

#     print(data['Prices'].describe())
#     print("-" * 40)
#     return data

# data1 = clean_price(data1)
# data2 = clean_price(data2)

# display(
#     data1.head(3),
#     data2.head(3),
# )




def clean_price(data_copy):
    data = data_copy.copy()
    
    # 1. Isolate the price string before any slashes
    data['Prices'] = data['Price'].str.split('/', n=1).str[0]
    
    # 2. Define a small inner function with your exact IF/ELSE logic for each row
    def row_logic(row_string):
        row_string = str(row_string)
        
        # Check if it's a USD listing
        is_usd = '$' in row_string
        
        # Clean out symbols and punctuation to extract the raw number
        clean_num_str = row_string.replace('₦', '').replace('$', '').replace(',', '').strip()
        
        try:
            val = float(clean_num_str)
        except ValueError:
            return 0  # Handle missing or broken values safely
        
        if is_usd and len(val) <= 5:  # Using 1M threshold to catch real $40k vs fake $30m
            return int(val * 1366.92)  # Multiply real USD
        else:
            return int(val)            # Else just strip and return the raw Naira value

    # 3. Use .apply() to execute your logic row-by-row down the column
    data['Prices'] = data['Prices'].apply(row_logic)
    
    # Filter to keep realistic records
    data = data[data['Prices'] >= 100000]
    #data.loc[data['Prices'] > 1000000000, 'Prices'] = (data['Prices'] / 1366.92).astype(int)
    
    print(data['Prices'].describe())
    print("-" * 40)
    
    return data

data1 = clean_price(data1)
data2 = clean_price(data2)

TypeError: object of type 'float' has no len()

In [12]:
locations = [
        "Abule Egba", "Agege", "Ajah", "Alimosho", "Amuwo Odofin", "Apapa", "Bariga",
        "Egbe Idimu", "Ejigbo", "Epe", "Gbagada", "Ibeju Lekki", "Iju", "Ikeja",
        "Ikorodu", "Ikotun Igando", "Ikoyi", "Ilupeju", "Ipaja", "Isolo", "Ketu",
        "Kosofe Ikosi", "Lagos Island", "Lekki", "Maryland", "Mushin", "Ogba",
        "Ogudu", "Ojo", "Ojodu", "Ojota", "Okota", "Oshodi", "Sangotedo", "Shomolu",
        "Surulere", "Victoria Island", "Yaba"
    ]

states = [
    "Abia", "Adamawa", "Akwa Ibom", "Anambra", "Bauchi", "Bayelsa",
    "Benue", "Borno", "Cross River", "Delta", "Ebonyi", "Edo", "Ekiti",
    "Enugu", "Gombe", "Imo", "Jigawa", "Kaduna", "Kano", "Katsina",
    "Kebbi", "Kogi", "Kwara", "Lagos", "Nasarawa", "Niger", "Ogun",
    "Ondo", "Osun", "Oyo", "Plateau", "Rivers", "Sokoto", "Taraba",
    "Yobe", "Zamfara", "Abuja"
]


def get_area(address):
    for area in locations:
        if pd.notna(address) and area.lower() in address.lower():
            return area
    parts = address.split(' ')
    if len(parts) >= 2:
        return parts[-2].strip()
    #return addres.strip()


def get_state(address):
    for state in states:
        if pd.notna(address) and state.lower() in address.lower():
            return state

    return 'Unknown'


data1['Location'] = data1['Address'].apply(get_area)
data1['State'] = data1['Address'].apply(get_state)

data2['Location'] = data2['Address'].apply(get_area)
data2['State'] = data2['Address'].apply(get_state)

data2['Location'] = data2['Location'].fillna(data2['Location'].mode()[0])

In [13]:
def property_category(data_copy):
    data = data_copy.copy()
    desc = data['Description'].str.lower()

    conditions = [
        desc.str.contains('mini flat', na=False),
        desc.str.contains('terraced|terrace', na=False),
        desc.str.contains('detached|semi detached|semi-detached|duplex', na=False),
        desc.str.contains('luxury', na=False),
        desc.str.contains('flat|apartment|block of flat', na=False),
        desc.str.contains('self contain|shared apartment|studio|self-contain|bq', na=False),
        desc.str.contains('office|commercial|shop|warehouse', na=False),
        desc.str.contains('land', na=False)
    ]

    categories = [
        'Mini Flat',
        'Self Contain / Shared Room',
        'Terraced Duplex',
        'Luxury Apartment',
        'Detached / Semi-Detached Duplex',
        'Standard Flat / Apartment',
        'Commercial Property',
        'Land'
    ]

    data['Property Type'] = np.select(conditions, categories, default = 'Other')

    return data

data1 = property_category(data1)
data2 = property_category(data2)

display(
    data1.head(3),
    data2.head(3)
)

,Title,Description,Address,Price,Toilets,Bedrooms,Bathrooms,Date Listed,Prices,Location,State,Property Type
0,Newly Renovated Room And Parlour Shortlet Apar...,1 BEDROOM MINI FLAT For Rent,11 Obamusa Agungi Lekki Lagos Agungi Lekki Lagos,"₦3,200,000/month",1,1,1,2026-06-10,3200000,Lekki,Lagos,Mini Flat
1,Unfurnished 3 Bedroom Apartment And A Bq,3 BEDROOM BLOCK OF FLATS FOR RENT,Fatai Arobieke Street Lekki Phase 1 Lekki Lagos,"₦23,000,000",4,3,3,2026-06-23,23000000,Lekki,Lagos,Detached / Semi-Detached Duplex
2,2 Bedroom Flat,2 BEDROOM MINI FLAT FOR RENT,Mutual Estate Eputu Ibeju Lekki Lagos,"₦2,500,000",3,2,3,2026-06-23,2500000,Ibeju Lekki,Lagos,Mini Flat


,Title,Description,Address,Price,Feature,Bedrooms,Bathrooms,Date Listed,Prices,Location,State,Property Type
0,Lovely Newly Renovated 2 Bedroom Flat,Lovely Newly Renovated 2 Bedroom Flat/Flat / A...,"Miracle Avenue, Opposite Wisebuyer Mall. Magbo...","₦ 1,500,000/year",2 Beds 1 Baths,2,1,2026-06-22,1500000,Owode,Ogun,Detached / Semi-Detached Duplex
1,Ultra Luxury 3 Bedroom Apartment Bq Pool Gym,Ultra Luxury 3 Bedroom Apartment Bq Pool Gym/F...,Old Ikoyi Old Ikoyi Ikoyi Lagos,"₦ 54,000,000",3 Beds 3 Baths,3,3,2026-06-19,54000000,Ikoyi,Lagos,Luxury Apartment
2,Mini Flat,Mini Flat/Flat Apartment FOR Rent,Lekki Phase 1 Lekki Lagos,"₦ 5,000,000/year",1 Beds 1 Baths,1,1,2026-03-07,5000000,Lekki,Lagos,Mini Flat


In [14]:
nature = ['Rent', 'Sale']

def get_nature(description):
    description_lower = str(description).lower()
    for item in nature:
        if item.lower() in description_lower:
            return item
    return 'Other'

data1['Nature'] = data1['Description'].apply(get_nature)
data2['Nature'] = data2['Description'].apply(get_nature)

In [15]:
data = [data1, data2]

for df_item in data:
    print(df_item['Nature'].value_counts())

Nature
Rent     6346
Sale      137
Other      18
Name: count, dtype: int64
Nature
Rent     12811
Sale       120
Other       12
Name: count, dtype: int64


In [16]:
def drop_col(data_copy):
    data = data_copy.copy()
    
    data = data[data['Nature'] == 'Rent']
    cols = ['Price', 'Description', 'Feature', 'Nature', 'Date_Info', 'Date Added']

    for col in cols:
        if col in data.columns:
            data = data.drop(columns = col)
            print(f'{len(cols)} columns dropped!')
        else:
            print(f'No such column: {col} found in dataset')
            break

    return data

data1 = drop_col(data1)
data2 = drop_col(data2)

display(
    data1.shape,
    data2.shape
)

6 columns dropped!
6 columns dropped!
No such column: Feature found in dataset
6 columns dropped!
6 columns dropped!
6 columns dropped!
6 columns dropped!
No such column: Date_Info found in dataset


(6346, 11)

(12811, 9)

In [17]:
order1 = ['Title', 'Address', 'Property Type', 'Location', 'State', 'Bedrooms', 'Bathrooms', 'Toilets', 'Date Listed', 'Prices']
order2 = ['Title', 'Address', 'Property Type', 'Location', 'State', 'Bedrooms', 'Bathrooms', 'Date Listed', 'Prices']

data1 = data1[order1]
data2 = data2[order2]

display(
    data1.sample(14, random_state = 42),
    data2.sample(14, random_state = 42)
)

,Title,Address,Property Type,Location,State,Bedrooms,Bathrooms,Toilets,Date Listed,Prices
801,2bedroom Flat At Ateere Kasumu,Ateere Kasumu Road Tipper Garage Akala Express...,Detached / Semi-Detached Duplex,Ibadan,Oyo,2,0,0,2026-06-23,1700000
6283,Spacious Studio Apartment,John Okafor Estate Agungi Lekki Lagos,Detached / Semi-Detached Duplex,Lekki,Lagos,1,0,1,2026-05-06,1800000
6276,Luxury 3 Bedroom Apartments,Old Ikoyi Ikoyi Lagos,Detached / Semi-Detached Duplex,Ikoyi,Lagos,3,3,4,2025-12-03,75000000
4766,Luxury Spacious Studio Apartment,Osapa London Lekki Lagos,Detached / Semi-Detached Duplex,Lekki,Lagos,1,1,1,2026-06-04,3000000
249,Spacious Self Gated 4bedroom Semidetached Dupl...,Ilasan Lekki Lagos,Terraced Duplex,Lekki,Lagos,4,4,5,2026-03-13,12000000
5839,Sharp 5 Bedroom Semi Detached Duplex,Osapa London Lekki Lagos,Terraced Duplex,Lekki,Lagos,5,5,6,2026-06-18,10000000
3949,Executive Finishef 2 Bedroom In The Heart Of A...,Off Jobowu Abule Egba Gba Abule Egba Abule Egb...,Detached / Semi-Detached Duplex,Abule Egba,Lagos,2,2,3,2025-11-24,2200000
4336,4 Bedroom Fully Detached House– Old Ikoyi,Old Ikoyi Ikoyi Lagos,Other,Ikoyi,Lagos,4,4,5,2026-02-06,60000000
6888,Fully Furnished 2 Bedroom Apartments For Lease,Ikate Ikate Elegushi Lekki Lagos,Detached / Semi-Detached Duplex,Lekki,Lagos,3,3,0,2025-05-30,18000000
3371,Spacious 1 Bedroom Apartment In A Serene Estate,Victoria Garden City Lekki Lagos,Mini Flat,Lekki,Lagos,1,1,1,2026-03-09,4500000


,Title,Address,Property Type,Location,State,Bedrooms,Bathrooms,Date Listed,Prices
1024,Brand New Luxury 2bedroom Apartment,Ologolo Ologolo Lekki Lagos,Luxury Apartment,Lekki,Lagos,2,2,2026-06-07,7000000
14483,2 Bedroom Flat,Victoria Crest 2 Orchid Rd Lafiaji Lekki Lagos,Detached / Semi-Detached Duplex,Lekki,Lagos,2,0,2026-05-08,5000000
435,Mini Flat,Unity Estates Badore Ajah Lagos,Mini Flat,Ajah,Lagos,0,0,2026-01-20,1500000
7396,Furnished 3 Bedroom Flat With Bq,Banana Island Ikoyi Lagos,Detached / Semi-Detached Duplex,Ikoyi,Lagos,3,3,2026-02-13,65000000
8,Semi Furnished Well Built 2 Bedroom Apartment,Ikota Lekki Lagos,Detached / Semi-Detached Duplex,Lekki,Lagos,2,3,2026-06-16,65000000
14814,Sharp Brand New Mini Flat,Off Market Square Ago Palace Okota Lagos,Mini Flat,Okota,Lagos,1,2,2026-06-02,2500000
855,2 Bedroom Flat,"Off Herbert Macaulay Way, Yaba Lagos",Detached / Semi-Detached Duplex,Yaba,Lagos,2,2,2026-04-07,5000000
3789,Spacious 1bedroom Apartment,Ado Road Ajah Lagos,Detached / Semi-Detached Duplex,Ajah,Lagos,1,1,2026-06-19,1700000
16014,A Newly Built Spacious & Serviced 3 Bedroom Flat,Close To Shoprite ( Video Is Available) Jabi A...,Detached / Semi-Detached Duplex,Jabi,Abuja,3,3,2026-06-04,23000000
17152,Newly Built 2 Bedroom Apartment,Evergreen Estate Aboru Iyana Ipaja Ipaja Lagos,Detached / Semi-Detached Duplex,Ipaja,Lagos,0,0,2026-01-04,2500000


In [18]:
data1.info(), data2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6346 entries, 0 to 7081
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Title          6346 non-null   object        
 1   Address        6346 non-null   object        
 2   Property Type  6346 non-null   object        
 3   Location       6346 non-null   object        
 4   State          6346 non-null   object        
 5   Bedrooms       6346 non-null   int64         
 6   Bathrooms      6346 non-null   int64         
 7   Toilets        6346 non-null   int64         
 8   Date Listed    6346 non-null   datetime64[ns]
 9   Prices         6346 non-null   int64         
dtypes: datetime64[ns](1), int64(4), object(5)
memory usage: 545.4+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 12811 entries, 0 to 17647
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Title

(None, None)

In [19]:
display(
    data1.describe(),
    data2.describe()
)

,Bedrooms,Bathrooms,Toilets,Date Listed,Prices
count,6346.000000,6346.000000,6346.000000,6346,6.346000e+03
mean,2.581311,2.495745,3.113615,2026-02-12 15:54:10.551528448,2.525851e+07
min,0.000000,0.000000,0.000000,2021-07-03 00:00:00,1.000000e+05
25%,1.000000,1.000000,1.000000,2026-01-08 00:00:00,3.500000e+06
50%,3.000000,3.000000,3.000000,2026-04-14 00:00:00,1.200000e+07
75%,4.000000,4.000000,5.000000,2026-05-28 00:00:00,3.200000e+07
max,10.000000,10.000000,20.000000,2026-09-27 00:00:00,9.773478e+08
std,1.601445,1.684135,2.001697,NaN,4.669564e+07


,Bedrooms,Bathrooms,Date Listed,Prices
count,12811.000000,12811.000000,12811,1.281100e+04
mean,1.687456,1.579502,2026-04-11 08:06:08.683162880,1.129799e+07
min,0.000000,0.000000,2023-08-10 00:00:00,1.000000e+05
25%,1.000000,0.000000,2026-03-15 00:00:00,2.000000e+06
50%,2.000000,1.000000,2026-04-30 00:00:00,4.000000e+06
75%,3.000000,3.000000,2026-06-02 00:00:00,1.000000e+07
max,10.000000,10.000000,2026-06-24 00:00:00,7.518060e+08
std,1.111173,1.258345,NaN,2.448242e+07


In [21]:
display(
    data1['Prices'].max(),
    data1['Prices'].min(),
    data2['Prices'].max(),
    data2['Prices'].min()
)

np.int64(977347800)

np.int64(100000)

np.int64(751806000)

np.int64(100000)

In [17]:
df = pd.concat([data1, data2])
df.shape

(19498, 10)

In [18]:
df.duplicated().sum()

np.int64(147)

In [19]:
df[df.duplicated()]

,Title,Address,Property Type,Location,State,Bedrooms,Bathrooms,Toilets,Date Listed,Prices
165,Luxury 2 Bedroom Maisonette With Excellent Fin...,Off Mobil Road Ilaje Lekki Scheme 2 Ajah Lagos,Other,Ajah,Lagos,2,2,3.0,2026-06-11,6500000
183,1bedroom (mini Flat).,Ologoloe Ologolo Lekki Lagos,Mini Flat,Lekki,Lagos,1,1,2.0,2026-03-31,3000000
264,New & Luxury 5 Bedroom Detached House + Bq+ Ro...,Banana Island Estate Banana Island Ikoyi Lagos,Terraced Duplex,Ikoyi,Lagos,5,5,6.0,2026-05-25,136555308
321,Fully Serviced 2 Bedroom Apartment,"Ikate, Costal Road Lekki Lagos",Other,Lekki,Lagos,2,2,3.0,2026-05-19,12000000
334,Luxury 3 Bedroom Apartment +bq+pool,Banana Island Estate Banana Island Ikoyi Lagos,Detached / Semi-Detached Duplex,Ikoyi,Lagos,3,3,4.0,2026-05-06,35000000
...,...,...,...,...,...,...,...,...,...,...
16274,Studio Apartment,Osapa London Lekki Lagos,Detached / Semi-Detached Duplex,Lekki,Lagos,1,1,NaN,2026-06-04,4000000
16659,Mini Flat,Oke Ira Ogba Lagos,Mini Flat,Ogba,Lagos,1,1,NaN,2026-04-24,1500000
16783,2 Bedroom Apartment,Osapa London Lekki Lagos,Detached / Semi-Detached Duplex,Lekki,Lagos,2,2,NaN,2026-05-24,10000000
16902,Mini Flat,Igbo Efon Lekki Lagos,Mini Flat,Lekki,Lagos,1,1,NaN,2026-04-11,3000000


In [20]:
df['Toilets'] = df['Toilets'].fillna(df['Bathrooms'])

df = df.drop_duplicates()
df.shape

(19347, 10)

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19347 entries, 0 to 17647
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Title          19347 non-null  object        
 1   Address        19347 non-null  object        
 2   Property Type  19347 non-null  object        
 3   Location       19347 non-null  object        
 4   State          19347 non-null  object        
 5   Bedrooms       19347 non-null  int64         
 6   Bathrooms      19347 non-null  int64         
 7   Toilets        19347 non-null  float64       
 8   Date Listed    19347 non-null  datetime64[ns]
 9   Prices         19347 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(5)
memory usage: 1.6+ MB


In [22]:
data1.to_csv('ng_properties.csv', index = False)
data2.to_csv('ng_realestates.csv', index = False)
df.to_csv('real_nig_properties.csv', index = False)

In [25]:
display(
    data1.describe(),
    data2.describe(),
    df.describe()
)

,Bedrooms,Bathrooms,Toilets,Date Listed,Prices
count,6529.000000,6529.000000,6529.000000,6529,6.529000e+03
mean,2.588911,2.487517,3.094961,2026-02-13 11:48:51.720018176,1.138068e+08
min,0.000000,0.000000,0.000000,2021-07-03 00:00:00,0.000000e+00
25%,1.000000,1.000000,1.000000,2026-01-08 00:00:00,3.500000e+06
50%,3.000000,3.000000,3.000000,2026-04-15 00:00:00,1.200000e+07
75%,4.000000,4.000000,5.000000,2026-05-29 00:00:00,3.500000e+07
max,10.000000,10.000000,20.000000,2026-09-27 00:00:00,1.025190e+11
std,1.625690,1.707970,2.026370,NaN,2.165572e+09


,Bedrooms,Bathrooms,Date Listed,Prices
count,12969.000000,12969.000000,12969,1.296900e+04
mean,1.703601,1.594649,2026-04-10 20:59:40.846634240,3.957963e+08
min,0.000000,0.000000,2023-08-10 00:00:00,0.000000e+00
25%,1.000000,0.000000,2026-03-14 00:00:00,2.000000e+06
50%,2.000000,1.000000,2026-04-30 00:00:00,4.000000e+06
75%,3.000000,3.000000,2026-06-02 00:00:00,1.100000e+07
max,10.000000,10.000000,2026-06-24 00:00:00,4.800000e+12
std,1.135439,1.282146,NaN,4.215127e+10


,Bedrooms,Bathrooms,Toilets,Date Listed,Prices
count,19347.000000,19347.000000,19347.000000,19347,1.934700e+04
mean,2.000310,1.894402,2.098206,2026-03-22 16:00:22.329043200,3.033029e+08
min,0.000000,0.000000,0.000000,2021-07-03 00:00:00,0.000000e+00
25%,1.000000,1.000000,1.000000,2026-02-28 00:00:00,2.500000e+06
50%,2.000000,2.000000,2.000000,2026-04-25 00:00:00,5.500000e+06
75%,3.000000,3.000000,3.000000,2026-06-01 00:00:00,2.000000e+07
max,10.000000,10.000000,20.000000,2026-09-27 00:00:00,4.800000e+12
std,1.385485,1.499910,1.724055,NaN,3.453368e+10
